In [29]:
import pulp

# Se inicializa el problema de Optimización declarando que es de Maximización
problema = pulp.LpProblem("Maximizacion_Ganancia_Fertilizantes", pulp.LpMaximize)

# a) Definición de las variables de decisión (con límite inferior 0 para la no negatividad)
# Al ser toneladas, las variables son continuas
x1 = pulp.LpVariable("X1_Fertilizante_F1", lowBound=0, cat='Continuous')
x2 = pulp.LpVariable("X2_Fertilizante_F2", lowBound=0, cat='Continuous')

# b) Formulación de la función objetivo
problema += 180 * x1 + 220 * x2, "Ganancia_Total"

# c) Planteamiento de las restricciones de disponibilidad de compuestos
problema += 3 * x1 + 4 * x2 <= 120, "Restriccion_Compuesto_A"
problema += 2 * x1 + 1 * x2 <= 80,  "Restriccion_Compuesto_B"
problema += 5 * x1 + 3 * x2 <= 150, "Restriccion_Compuesto_C"

# Se resuelve el modelo
problema.solve()

# resultados
print("RESULTADOS DE LA OPTIMIZACIÓN")
print(f"Estado de la solución: {pulp.LpStatus[problema.status]}")
print(f"Producir Fertilizante F1 (X1): {x1.varValue} toneladas")
print(f"Producir Fertilizante F2 (X2): {x2.varValue} toneladas")
print(f"Ganancia Máxima Semanal: ${pulp.value(problema.objective)}")

RESULTADOS DE LA OPTIMIZACIÓN
Estado de la solución: Optimal
Producir Fertilizante F1 (X1): 21.818182 toneladas
Producir Fertilizante F2 (X2): 13.636364 toneladas
Ganancia Máxima Semanal: $6927.27284


In [30]:
import sympy as sp

def simular_produccion(semanas_totales, S_0=200, M_0=80):
    S_n, M_n = S_0, M_0
    print(f"Semana 0 -> Sillas: {S_n} | Mesas: {M_n}")

    for n in range(1, semanas_totales + 1):
        S_next = 0.6 * S_n + 0.2 * M_n + 40
        M_next = 0.1 * S_n + 0.5 * M_n + 20
        S_n, M_n = S_next, M_next
        print(f"Semana {n} -> Sillas: {round(S_n, 1)} | Mesas: {round(M_n, 1)}")

    return S_n, M_n

# Simulación de producción
cantidad_semanas = 2
print("SIMULACIÓN DE PRODUCCIÓN")
simular_produccion(cantidad_semanas)

# Valores de equilibrio
S_eq, M_eq = sp.symbols('S_eq M_eq')

ecuacion_sillas = sp.Eq(S_eq, 0.6 * S_eq + 0.2 * M_eq + 40)
ecuacion_mesas = sp.Eq(M_eq, 0.1 * S_eq + 0.5 * M_eq + 20)

solucion = sp.solve((ecuacion_sillas, ecuacion_mesas), (S_eq, M_eq))

print("\nVALORES DE EQUILIBRIO")
print(f"Ecuación Sillas: {ecuacion_sillas}")
print(f"Ecuación Mesas:  {ecuacion_mesas}")
print(f"Tendencia: Sillas = {solucion[S_eq]:.2f}, Mesas = {solucion[M_eq]:.2f}")

SIMULACIÓN DE PRODUCCIÓN
Semana 0 -> Sillas: 200 | Mesas: 80
Semana 1 -> Sillas: 176.0 | Mesas: 80.0
Semana 2 -> Sillas: 161.6 | Mesas: 77.6

VALORES DE EQUILIBRIO
Ecuación Sillas: Eq(S_eq, 0.2*M_eq + 0.6*S_eq + 40)
Ecuación Mesas:  Eq(M_eq, 0.5*M_eq + 0.1*S_eq + 20)
Tendencia: Sillas = 133.33, Mesas = 66.67


In [31]:
import numpy as np
import sympy as sp
from scipy.integrate import odeint

# Parámetros del modelo SIS
N = 10000
beta = 0.00003
gamma = 0.1

# PUNTOS DE EQUILIBRIO
I_eq = sp.symbols('I_eq')

# Planteamos la ecuación dI/dt = 0
ecuacion_equilibrio = sp.Eq(beta * (N - I_eq) * I_eq - gamma * I_eq, 0)

# SymPy resuelve la ecuación automáticamente
puntos_equilibrio = sp.solve(ecuacion_equilibrio, I_eq)

print("PUNTOS DE EQUILIBRIO")
for i in range(len(puntos_equilibrio)):
    print(f"Equilibrio {i+1}: {round(puntos_equilibrio[i])} infectados")

# SIMULACIÓN CON SCIPY
def ecuacion_epidemia(I, t):
    return beta * (N - I) * I - gamma * I

I_inicial = 50
dias = np.linspace(0, 100, 11)

infectados_simulacion = odeint(ecuacion_epidemia, I_inicial, dias)

print("\nRESULTADOS DE LA SIMULACIÓN")
for i in range(len(dias)):
    dia_actual = round(dias[i])
    infectados_actuales = round(infectados_simulacion[i][0])
    print(f"Día {dia_actual:>3}: {infectados_actuales} infectados")

PUNTOS DE EQUILIBRIO
Equilibrio 1: 0 infectados
Equilibrio 2: 6667 infectados

RESULTADOS DE LA SIMULACIÓN
Día   0: 50 infectados
Día  10: 353 infectados
Día  20: 1947 infectados
Día  30: 5020 infectados
Día  40: 6383 infectados
Día  50: 6627 infectados
Día  60: 6661 infectados
Día  70: 6666 infectados
Día  80: 6667 infectados
Día  90: 6667 infectados
Día 100: 6667 infectados


In [32]:
import numpy as np

# PARÁMETROS DEL MODELO
K = 1000  # Capacidad máxima de la laguna
H = 80    # Extracción anual (pesca)
r = 0.4   # Tasa de crecimiento (CORREGIDA)
P = 400   # Población inicial (Año 0)

# CÁLCULO DE AÑOS 1 Y 2
print("EVOLUCIÓN DE LA POBLACIÓN")
print("Año 0 :", P, "peces")

# Ciclo básico para proyectar la población
for n in range(1, 3):
    # Fórmula del modelo logístico discreto
    P_siguiente = P + r * P * (1 - P / K) - H

    # Actualizamos el valor para el próximo ciclo
    P = P_siguiente

    print("Año", n, ":", round(P, 2), "peces")

# CÁLCULO DE EQUILIBRIOS
print("\nPUNTOS DE EQUILIBRIO")
# En el equilibrio: r*P*(1 - P/K) - H = 0
# Organizado como ecuación cuadrática (aP^2 + bP + c = 0):
a = -r / K
b = r
c = -H

# Fórmula cuadrática (-b ± raíz(b^2 - 4ac)) / 2a
discriminante = (b**2) - (4 * a * c)

equilibrio_1 = (-b + np.sqrt(discriminante)) / (2 * a)
equilibrio_2 = (-b - np.sqrt(discriminante)) / (2 * a)

# Identificamos cuál es el estable (el mayor) y el inestable (el menor)
eq_inestable = min(equilibrio_1, equilibrio_2)
eq_estable = max(equilibrio_1, equilibrio_2)

print("Equilibrio Inestable (Umbral de supervivencia) :", round(eq_inestable, 2), "peces")
print("Equilibrio Estable (Capacidad con pesca)       :", round(eq_estable, 2), "peces")

EVOLUCIÓN DE LA POBLACIÓN
Año 0 : 400 peces
Año 1 : 416.0 peces
Año 2 : 433.18 peces

PUNTOS DE EQUILIBRIO
Equilibrio Inestable (Umbral de supervivencia) : 276.39 peces
Equilibrio Estable (Capacidad con pesca)       : 723.61 peces
